# Composition Audit

Static analysis for reward composition configs. Give it your setup and it checks for structural problems before training starts.

In [1]:
import sys
sys.path.insert(0, '..')
import numpy as np
from src.composition import RewardComponent, ComponentType
from src.audit import audit, print_report

## A typical production config (before audit)

In [2]:
rng = np.random.RandomState(42)
config = [
    RewardComponent(name="correctness", fn=lambda a: float(a>0.4)*np.exp(-5*(a-0.7)**2),
                    component_type=ComponentType.SCORER, weight=1.0),
    RewardComponent(name="format_compliance", fn=lambda a: 1.0/(1.0+np.exp(-15*(a-0.2))),
                    component_type=ComponentType.SCORER, weight=0.5),
    RewardComponent(name="style", fn=lambda a: np.clip(np.exp(-15*(a-0.5)**2)+rng.normal(0,0.2), 0, 1),
                    component_type=ComponentType.SCORER, weight=0.3, independent_of=["correctness"]),
    RewardComponent(name="safety", fn=lambda a: 0.9 if a>0.05 else 0.0,
                    component_type=ComponentType.SCORER, weight=0.5),
]
print_report(audit(config, n_samples=2000, seed=42))

COMPOSITION AUDIT
4 components (0 gates, 4 scorers)

  [FAIL] gate typing
         ['format_compliance', 'safety'] look like prerequisites but are typed as scorers. This creates priority inversion risk. Make them gates.

  [WARN] variance balance
         5.1x variance ratio. 'correctness' may have outsized influence.

  [OK  ] independence
         1 pair(s) checked, all clean

  [WARN] health monitoring
         No validators set. Signal degradation will go undetected.

  [OK  ] reward range
         Range [0.02, 2.25]

-------------------------------------------------------
  [FAIL] gate typing - ['format_compliance', 'safety'] look like prerequisites but are typed as scorers. This creates priority inversion risk. Make them gates.
  [WARN] variance balance - 5.1x variance ratio. 'correctness' may have outsized influence.
  [WARN] health monitoring - No validators set. Signal degradation will go undetected.


## After applying recommendations

In [3]:
def safety_check(scores, step):
    if len(scores) < 50: return True
    return sum(1 for s in scores[-50:] if s > 0.85)/50 < 0.95

fixed = [
    RewardComponent(name="correctness", fn=lambda a: float(a>0.4)*np.exp(-5*(a-0.7)**2),
                    component_type=ComponentType.SCORER, weight=1.0, influence_cap=1.5),
    RewardComponent(name="format_compliance", fn=lambda a: 1.0/(1.0+np.exp(-15*(a-0.2))),
                    component_type=ComponentType.GATE, gate_threshold=0.5),
    RewardComponent(name="style", fn=lambda a: np.clip(np.exp(-15*(a-0.5)**2)+np.random.normal(0,0.2), 0, 1),
                    component_type=ComponentType.SCORER, weight=0.3, influence_cap=1.2, independent_of=["correctness"]),
    RewardComponent(name="safety", fn=lambda a: 0.9 if a>0.05 else 0.0,
                    component_type=ComponentType.GATE, gate_threshold=0.5, validator=safety_check),
]
print_report(audit(fixed, n_samples=2000, seed=42))

COMPOSITION AUDIT
4 components (2 gates, 2 scorers)

  [OK  ] gate typing
         2 gate(s), 2 scorer(s)

  [OK  ] variance balance
         1.6x ratio (acceptable)

  [OK  ] independence
         1 pair(s) checked, all clean

  [OK  ] health monitoring
         Validators on: ['safety']

  [OK  ] reward range
         Range [0.05, 3.20]

